In [ ]:
'''
Simple MPL
Autor: Maria Hadam
28.06.2026
Edited: Ziemowit Olinkiewicz
20.07.2026
'''

import json
import torch
import sys
sys.path.append("./source")

import model as mlp
import numpy as np
import lindblad_solver as solver
import dataset as ds
import training as t
import plotting as plt

In [ ]:
# Load the dataset

root = "data/dataset_1_rel20_decoh40_5000t"     # name of the dataset directory
# root = "data/dataset_2_rel40_decoh20_5000t"

dataset = ds.TrajectoryDataset(root)
x, y, rho_t, rho_tp1 = dataset[0]                # load the first trajectory
print(f"number of trajectories: {len(dataset)}")
print(f"feature shape (input): {x.shape}  (n_steps+1={dataset._n_steps_plus_1}, 32 per step)")
print(f"label shape: {y.shape}  -> {ds.TAU_KEYS}")
print(f"label of the first trajectory (log-tau): {y}")
print(f"rho_t shape: {rho_t.shape}, rho_tp1 shape: {rho_tp1.shape}  (random consecutive pair)")
print()

# Preprocessing: prepare the dataset for training
train_ds, val_ds, test_ds = ds.split_dataset(dataset)   # split into train/val/test
print(f"split sizes: train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

mean, std, labels = ds.compute_label_stats(dataset, train_ds.indices)
print(f"train label mean: {mean}")
print(f"train label std:  {std}")
#print(f"train label:  {labels}")

In [ ]:
input_dim = dataset.feature_dim
output_dim = 5
model = mlp.MLP(input_dim, output_dim, hidden_dims=[256, 128, 64, 32], activation='relu', use_batchnorm=True, dropout_rate=0.2)
print(model)

dummy_x = torch.randn(8, input_dim)  # batch of 8
out = model(dummy_x)
print("output shape:", out.shape)  # expect (8, 5)
assert out.shape == (8, output_dim)
print("OK")

In [ ]:
pinn, dataset, (train_ds, val_ds, test_ds), history = t.train_pinn(
    root=root,
    n_epochs=10,
    batch_size=64,
    lr=1e-3,
    warmup_epochs=10,
    min_lambda_phys=0,#1e4,
    max_lambda_phys=0,#1e8,
    lambda_data=1.0,
    hidden_dims=(3208, 802, 128, 64, 32),
    num_workers=4,
    checkpoint_path="best_pinn_dataset1.pt",
    seed=0,
)

print("Training complete. Best checkpoint loaded.")

# save history so you can replot later without retraining
import json
with open("history_dataset1.json", "w") as f:
    json.dump(history, f)

plt.plot_history(history, save_path="pinn_training_dataset3.png")

In [ ]:
def traj_to_real_features(rhos: np.ndarray, times, timestep: float) -> np.ndarray:
    """
    Convert a complex density-matrix trajectory into a flat real feature vector,
    keeping only density matrices separated by at least `timestep`.

    Parameters
    ----------
    rhos : np.ndarray
        Complex array of shape (n_steps+1, 4, 4).
    times : array-like
        Time corresponding to each density matrix.
    timestep : float
        Minimum time separation between selected density matrices.

    Returns
    -------
    np.ndarray
        Flattened float64 feature vector containing the real and imaginary parts
        of the selected density matrices.
    """
    times = np.asarray(times)

    if len(rhos) != len(times):
        raise ValueError("rhos and times must have the same length.")

    # Always keep the first density matrix
    indices = [0]
    last_time = times[0]

    for i in range(1, len(times)):
        if times[i] - last_time >= timestep:
            indices.append(i)
            last_time = times[i]

    rhos_selected = rhos[indices]

    real_part = rhos_selected.real.astype(np.float64)
    imag_part = rhos_selected.imag.astype(np.float64)

    stacked = np.stack([real_part, imag_part], axis=-1)
    return stacked.reshape(-1)

In [ ]:
#kawalek kodu do wyciagania wynikow z ML i porownywania z prawdziwymi labelkami

features, labels, rho_t, rho_tp1 = dataset.__getitem__(101)
taus_pred_log, taus_pred, gammas_pred = pinn.forward(features)
print(labels)
print(torch.log(taus_pred))

print(torch.exp(labels))
print(taus_pred)